# 28 · Fill the missing MNIST comparison rows

Extend the original 20-image-gallery table using the exact Stage 6 test sources, candidates and transformations. Add a conventional quadratic-gradient structure tensor, zero-mean normalized cross-correlation, block-normalized HOG, and frozen ImageNet features. Comparisons distinguish the alignment policy from the field or descriptor.

The original Stage 6 rows reconcile exactly. The strongest new aligned comparator is the conventional normalized structure tensor, which scores 97.5% exact-source and 43.8% same-digit on the pilot.

In [1]:
from pathlib import Path
import sys,json
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'observables').is_dir())
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from observables.stage8 import BASELINES,FROZEN,PRIMARY,VIEWS,contrast
OUT=ROOT/'results'/'stage8'
plt.rcParams.update({'figure.dpi':110,'figure.constrained_layout.use':True,
                     'axes.spines.top':False,'axes.spines.right':False})
def show(name):
    plt.savefig(OUT/(name+'.png'),dpi=150)
    plt.show()
summary=pd.read_csv(OUT/'summary.csv')
decisions=pd.read_csv(OUT/'decisions.csv.gz')

Matplotlib is building the font cache; this may take a moment.


## Classical baselines and alignment
Pixels, magnitude-weighted local Q, the quadratic-gradient tensor and ZNCC compare within a fixed canvas and over the same 49 open shifts. Search requires 65% shared support. Each aligned representation chooses its own best offset.

The tensor baseline averages gradient outer products in a 5 × 5 neighborhood. We use its normalized traceless part, with a small trace floor. This keeps the same spatial scale as local Q while changing its gradient weighting and normalization. ZNCC centers both images on the shared overlap. The block-normalized scikit-image HOG uses 9 bins, 7 × 7 cells, 2 × 2 blocks and L2-Hys normalization. A second HOG row computes HOG on the shared support after choosing the ZNCC offset. That is a two-step baseline, with no separate HOG search.

In [3]:
from observables.open_images import Observation,translate
from observables.stage8 import Classical,zncc_distance
a=np.zeros((28,28));a[6:20,9:13]=.8;a[16:20,9:19]=1
shifted=translate(Observation.full(a),(2,-3))
scores=Classical(a[None]).compare(shifted)[0]
display(pd.Series(scores,index=BASELINES,name='controlled distance').round(10))
assert scores[list(BASELINES).index('aligned ZNCC')]<1e-10
mask=np.ones(a.shape,dtype=bool)
assert zncc_distance(a,.6*a+.2,mask)<1e-10
assert scores[list(BASELINES).index('HOG after ZNCC')]<1e-10

ZNCC                        0.389417
aligned ZNCC                0.000000
standard HOG                0.399415
HOG after ZNCC              0.000000
structure tensor            0.537193
aligned structure tensor    0.000000
Name: controlled distance, dtype: float64

## Pilot comparison
The frozen rows use one cosine score on a 56 × 56 full-frame embedding. They use no translation search. Notebook 30 compares mean and second-order pooling on their intermediate feature maps. The table preserves distinct policies. No result from another dataset is imported into a row.

In [5]:
pilot=summary.query("panel=='pilot' and gallery_type=='full'").set_index('method')
original=pd.read_csv(ROOT/'results'/'stage6'/'transformed_summary.csv',index_col=0)
pd.testing.assert_frame_equal(pilot.loc[original.index,original.columns],original,check_dtype=False)
methods=['pixels','aligned pixels','local Q','aligned local Q','aligned Q magnitude',
         *BASELINES,'ResNet-50 cosine','EfficientNet-B0 cosine']
display((100*pilot.loc[methods,['identity','class']]).round(2))
print('Stage 6 original rows reconciled exactly.')

,identity,class
method,,
pixels,75.90,30.31
aligned pixels,95.94,37.50
local Q,90.12,46.60
aligned local Q,97.73,45.27
aligned Q magnitude,96.33,34.34
ZNCC,74.53,31.17
aligned ZNCC,95.23,39.88
standard HOG,85.39,35.23
HOG after ZNCC,91.91,36.64


Stage 6 original rows reconciled exactly.


Every row uses the same query transformations and candidate identities. Geometry still varies by method. Classical baselines share a declared search budget where applicable. Fixed frozen-encoder comparisons measure the selected checkpoints and small-image preprocessing, and should be labeled as such in a manuscript.